# NexusAI Core-01 — Google Colab Training Notebook

**Our own model. Our own data. No external AI APIs.**

This notebook trains **NexusAI Core-01** (8,842,560 params, vocab 8,000, ctx 512)
on a single NVIDIA T4 via Google Colab + Google Drive persistence.

| Stage | Accelerator | Approval flag |
|---|---|---|
| Env / Drive / package / tokenizer / dataset checks | CPU-side checks | none |
| GPU verify + VRAM probe | GPU | none |
| Smoke test (~200 steps, real data) | GPU | `APPROVE_SMOKE_TEST` |
| Full pretraining (~100M tokens) | GPU | `APPROVE_FULL_TRAINING` |

**Rules:** enable flags ONE AT A TIME after reading the printed plan.
Colab sessions end after ~12h — checkpoints go to Google Drive so training can resume next session.


## Cell 2 — Environment Inspection (read-only)

In [ ]:
import os, sys, platform, shutil, subprocess
print("python:", sys.version.split()[0], "|", platform.platform())
import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu:", p.name, "| vram %.2f GB" % (p.total_memory/1e9),
          "| capability", p.major, p.minor)
for mod, pip in [("sentencepiece","sentencepiece"),("pyarrow","pyarrow"),("numpy","numpy")]:
    try:
        m=__import__(mod); print(mod, getattr(m,"__version__","?"))
    except ImportError:
        print(mod, "MISSING -> !pip install -q", pip)
for d in ["/content/drive/MyDrive", "/content"]:
    try:
        u=shutil.disk_usage(d); print("disk",d,"free %.1f GB"%(u.free/1e9))
    except Exception as e: print("disk",d,"?",e)
print("cpu_count:", os.cpu_count())
try:
    import psutil; print("ram %.1f GB"%(psutil.virtual_memory().total/1e9))
except Exception: pass

## Cell 3 — Google Drive Mount & Paths

Mounts Drive and defines all working paths. Persistent data lives under
`/content/drive/MyDrive/NexusAI/`; runtime scratch under `/content/work`.
See `NEXUSAI_CORE01_COLAB_SETUP.md` for the exact upload layout.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/NexusAI"
PACKAGE_DIR = os.path.join(DRIVE_ROOT, "package")          # contains the ZIP or extracted folder
DATA_DIR    = os.path.join(DRIVE_ROOT, "data")             # english_*.jsonl + manifest
TOKENIZER_PATH = None   # resolved in Cell 5
PREP_SCRIPT = None
WORK = "/content/work/nexusai_core01"
PKG_EXTRACT = os.path.join(WORK, "pkg")
CKPT_SMOKE  = os.path.join(DRIVE_ROOT, "checkpoints_smoke")
CKPT_FULL   = os.path.join(DRIVE_ROOT, "checkpoints_full")
LOG_DIR     = os.path.join(DRIVE_ROOT, "logs")

for d in [WORK, PKG_EXTRACT, CKPT_SMOKE, CKPT_FULL, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("PACKAGE_DIR exists:", os.path.isdir(PACKAGE_DIR), "->", os.listdir(PACKAGE_DIR) if os.path.isdir(PACKAGE_DIR) else "MISSING — upload package first")
print("DATA_DIR exists:", os.path.isdir(DATA_DIR), "->", os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "MISSING — upload corpus first")

## Cell 4 — Package Integrity & Extraction

Finds `nexusai_core01_kaggle_package.zip` (or an already-extracted folder) in the
Drive package dir, verifies SHA-256 against the known-good hash, extracts to
runtime `/content/work`, and locates `core01/`, `models/`, tokenizer, prep script.
`PACKAGE_OK=False` blocks later cells.

In [ ]:
import glob, zipfile, hashlib, shutil

PACKAGE_OK = False
PACKAGE_ROOT = None
EXPECTED_SHA = "1b0c66f03457eb764b08dacde5594e4093581b2b3f116ad9e3176a97d33be970"
EXPECTED_FILES = 15

try:
    zips = glob.glob(os.path.join(PACKAGE_DIR, "**/*.zip"), recursive=True)
    dirs = glob.glob(os.path.join(PACKAGE_DIR, "**/nexusai_core01_kaggle_package"), recursive=True)
    print("zip candidates:", zips); print("dir candidates:", dirs)
    if zips:
        zpath = zips[0]
        h = hashlib.sha256()
        with open(zpath,"rb") as f:
            for c in iter(lambda: f.read(1<<20), b""): h.update(c)
        print("zip sha256:", h.hexdigest())
        print("sha256 match:", h.hexdigest() == EXPECTED_SHA)
        if os.path.exists(PKG_EXTRACT): shutil.rmtree(PKG_EXTRACT)
        os.makedirs(PKG_EXTRACT, exist_ok=True)
        with zipfile.ZipFile(zpath) as z:
            names = z.namelist(); z.extractall(PKG_EXTRACT)
        print("zip files:", len(names), "(expected", EXPECTED_FILES, ")")
    elif dirs:
        print("using extracted dir (sha check skipped)")
        if os.path.exists(PKG_EXTRACT): shutil.rmtree(PKG_EXTRACT)
        shutil.copytree(dirs[0], PKG_EXTRACT)
    else:
        raise FileNotFoundError("No package found — upload the ZIP to Drive NexusAI/package/ first")

    roots = [d for d in glob.glob(PKG_EXTRACT+"/**/*", recursive=True)
             if os.path.isdir(os.path.join(d,"core01")) and os.path.isdir(os.path.join(d,"models"))]
    assert roots, "core01/+models/ not found"
    PACKAGE_ROOT = roots[0]
    for req in ["core01/__init__.py","core01/transformer.py","core01/training.py",
                "core01/dataset.py","core01/pretraining_driver.py",
                "models/nexusai_core_01/config/model_config.py"]:
        assert os.path.exists(os.path.join(PACKAGE_ROOT,req)), "missing "+req
    toks = glob.glob(PACKAGE_ROOT+"/**/sp_bpe_8k.model", recursive=True)
    assert toks, "tokenizer missing"
    TOKENIZER_PATH = toks[0]
    preps = glob.glob(PACKAGE_ROOT+"/**/prepare_english_corpus.py", recursive=True)
    PREP_SCRIPT = preps[0] if preps else None
    sys.path.insert(0, PACKAGE_ROOT)
    sys.path.insert(0, os.path.join(PACKAGE_ROOT,"models"))
    PACKAGE_OK = True
    print("PACKAGE_ROOT:", PACKAGE_ROOT)
    print("TOKENIZER_PATH:", TOKENIZER_PATH)
    print("PACKAGE_OK =", PACKAGE_OK)
except Exception as e:
    print("PACKAGE INIT FAILED:", repr(e))
    print("Fix: upload nexusai_core01_kaggle_package.zip to Drive NexusAI/package/ and re-run.")

## Cell 5 — Tokenizer Verification

Loads the real `sp_bpe_8k` tokenizer via the package's `NexusAITokenizer`.
Expects vocab 8,000, IDs pad=0/unk=1/bos=2/eos=3, encode/decode round-trip.

In [ ]:
assert PACKAGE_OK, "Package init failed — fix Cell 4 first."
from core01.tokenizer_interface import NexusAITokenizer
from nexusai_core_01.config.model_config import NexusAICore01Config

cfg = NexusAICore01Config()
tok = NexusAITokenizer(TOKENIZER_PATH, cfg)
vocab = tok.get_vocab_size()
print("vocab_size:", vocab, "(expected 8000)")
print("special IDs: pad=%d unk=%d bos=%d eos=%d" % (tok.pad_id,tok.unk_id,tok.bos_id,tok.eos_id))
assert vocab == 8000 and (tok.pad_id,tok.unk_id,tok.bos_id,tok.eos_id)==(0,1,2,3)
h=hashlib.sha256()
with open(TOKENIZER_PATH,"rb") as f:
    for c in iter(lambda:f.read(1<<20),b""): h.update(c)
print("tokenizer sha256:", h.hexdigest())
ids = tok.encode("Tokenizer verification test for NexusAI Core-01.", add_special_tokens=True)
print("encode ok, len:", len(ids), "| decode:", tok.decode(ids)[:60])
TOKENIZER_OK = True

## Cell 6 — Dataset Integrity Verification

Checks the prepared corpus in Drive `NexusAI/data/`: `english_train.jsonl`,
`english_val.jsonl`, `manifest.json`. Verifies real token counts, zero
train/val overlap, and driver `load_texts` compatibility.
If files are missing: upload them from Kaggle Output or re-run corpus prep
(see setup doc). `DATASET_OK=False` blocks training.

In [ ]:
DATASET_OK = False
train_path = os.path.join(DATA_DIR, "english_train.jsonl")
val_path   = os.path.join(DATA_DIR, "english_val.jsonl")
man_path   = os.path.join(DATA_DIR, "manifest.json")

if not all(os.path.exists(p) for p in [train_path,val_path,man_path]):
    print("MISSING dataset files. Needed in", DATA_DIR)
    print("  english_train.jsonl, english_val.jsonl, manifest.json")
    print("Get them from Kaggle Output download OR re-run prep (setup doc).")
else:
    import json
    tr = [json.loads(l) for l in open(train_path,encoding="utf-8") if l.strip()]
    va = [json.loads(l) for l in open(val_path,encoding="utf-8") if l.strip()]
    tr_tok = sum(d["ntok"] for d in tr); va_tok = sum(d["ntok"] for d in va)
    print(f"train: {len(tr):,} docs / {tr_tok:,} tokens")
    print(f"val:   {len(va):,} docs / {va_tok:,} tokens")
    overlap = len({d["text"] for d in tr} & {d["text"] for d in va})
    print("overlap docs:", overlap, "| empty:", sum(1 for d in tr+va if not d["text"].strip()))
    man = json.load(open(man_path,encoding="utf-8"))
    print("manifest tokens:", man["target"]["actual_accepted_tokens"])
    from core01.pretraining_driver import load_texts
    docs = load_texts(train_path)
    print("driver load_texts:", len(docs), "docs")
    DATASET_OK = (overlap==0 and tr_tok+va_tok>0 and len(docs)==len(tr))
    print("DATASET_OK:", DATASET_OK)

## Cell 7 — Model Architecture & VRAM Probe

Initializes Core-01 on `cuda:0`, prints parameter count (must be 8,842,560),
and runs one dummy forward+backward to estimate memory. No optimizer step —
this is a probe, not training.

In [ ]:
assert PACKAGE_OK and TOKENIZER_OK, "fix earlier cells first"
import torch
from core01.transformer import NexusAICore01

device = "cuda" if torch.cuda.is_available() else "cpu"
model = NexusAICore01(cfg).to(device)
n_params = model.count_parameters()
print("params:", n_params, "(expected 8,842,560)")
assert n_params == 8_842_560

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    model.train()
    x = torch.randint(0, cfg.vocab_size, (2, cfg.max_len), device=device)
    y = torch.randint(0, cfg.vocab_size, (2, cfg.max_len), device=device)
    with torch.autocast("cuda", dtype=torch.float16):
        out = model(x, targets=y)
    out["loss"].backward()
    peak = torch.cuda.max_memory_allocated()/1e9
    print("probe loss:", float(out["loss"]), "| peak VRAM %.2f GB" % peak)
    del model, x, y, out
    torch.cuda.empty_cache()
    print("VRAM headroom OK for smoke; adjust batch if probe > ~10 GB")
else:
    print("CPU only — smoke/full need GPU. Set Runtime > Change runtime type > T4.")
MODEL_OK = (n_params == 8_842_560)

## Cell 8 — Training Configuration Review

Shows the exact driver command that Cell 9 will run. Review every value.
Driver = `core01/pretraining_driver.py` (unchanged production code).
Single GPU `cuda:0`, FP16 AMP, packed sequences, atomic checkpoints to Drive.

In [ ]:
assert PACKAGE_OK and DATASET_OK, "fix earlier cells first"
DRIVER = os.path.join(PACKAGE_ROOT, "core01", "pretraining_driver.py")
print("driver:", DRIVER, "| exists:", os.path.exists(DRIVER))

SMOKE_CMD = [sys.executable, DRIVER,
    "--train-file", train_path, "--val-file", val_path,
    "--output-dir", CKPT_SMOKE, "--tokenizer", TOKENIZER_PATH,
    "--device", "cuda", "--precision", "fp16",
    "--batch-size", "32", "--gradient-accumulation-steps", "1",
    "--learning-rate", "3e-4", "--warmup-steps", "100",
    "--epochs", "1", "--max-steps", "200",
    "--checkpoint-interval", "100", "--eval-interval", "100",
    "--log-interval", "10"]
print("SMOKE plan:")
print(" ".join(SMOKE_CMD))
print()
print("Full run will use same driver WITHOUT --max-steps, --epochs 1-3,")
print("checkpoint dir ->", CKPT_FULL, "(Drive-persisted)")

## Cell 9 — GPU Smoke Test — **OWNER APPROVAL GATE**

`APPROVE_SMOKE_TEST=False` by default. When True: runs ~200 real optimizer
steps on the actual corpus with FP16 AMP, saves checkpoints to Drive.
This validates the pipeline — it does NOT mean the model is trained.
If it OOMs: lower `--batch-size` / raise `--gradient-accumulation-steps`.

In [ ]:
APPROVE_SMOKE_TEST = False   # <-- owner sets True after reviewing Cell 8

SMOKE_DONE = False
if not APPROVE_SMOKE_TEST:
    print("SMOKE BLOCKED: APPROVE_SMOKE_TEST=False (default).")
    print("Review Cell 8 command, then set the flag True and re-run.")
else:
    assert PACKAGE_OK and TOKENIZER_OK and DATASET_OK and MODEL_OK,         "preconditions failed"
    assert torch.cuda.is_available(), "no GPU — change runtime to T4 first"
    print("running smoke:", " ".join(SMOKE_CMD))
    import subprocess
    r = subprocess.run(SMOKE_CMD, cwd="/content/work")
    print("smoke exit:", r.returncode)
    SMOKE_DONE = (r.returncode == 0)
    print("SMOKE_DONE:", SMOKE_DONE)
    print("Check output above for: finite loss, val_loss, checkpoint saved.")

## Cell 10 — Smoke Test Evaluation

Read-only review of smoke results. Paste/verify: initial vs latest train
loss, val_loss, tokens/sec, peak VRAM, checkpoint files created.
Only proceed to full training if loss decreased and no NaN/Inf appeared.

In [ ]:
# Read-only evaluation of the smoke run artifacts.
smoke_files = sorted(glob.glob(os.path.join(CKPT_SMOKE, "*.pt"))) if SMOKE_DONE else []
print("smoke checkpoints:", [os.path.basename(p) for p in smoke_files])
for p in smoke_files[:3]:
    print(" ", os.path.basename(p), "%.1f MB" % (os.path.getsize(p)/1e6))
if smoke_files:
    import torch as _t
    ck = _t.load(smoke_files[-1], map_location="cpu", weights_only=False)
    print("ckpt keys sample:", [k for k in ck.keys()][:10])
    print("model_status:", ck.get("model_status"))
    print("step:", ck.get("step"), "| epoch:", ck.get("epoch"))
print()
print("REVIEW: did train loss go DOWN? any NaN? val_loss finite?")
print("If yes -> you may approve Cell 11. If no -> diagnose before continuing.")

## Cell 11 — Full Pretraining — **OWNER APPROVAL GATE**

`APPROVE_FULL_TRAINING=False` by default. Requires `SMOKE_DONE=True`.
Runs the real pretraining: ~98.5M train tokens, FP16, single T4.
Checkpoints every 500 steps to Drive; resume-safe across sessions.
Session limit ~12h — if cut off, next session resumes via `--resume`.

In [ ]:
APPROVE_FULL_TRAINING = False   # <-- owner sets True ONLY after smoke review

FULL_DONE = False
if not APPROVE_FULL_TRAINING:
    print("FULL TRAINING BLOCKED: APPROVE_FULL_TRAINING=False (default).")
    print("Requirements: SMOKE_DONE=True + owner sign-off on config.")
else:
    assert SMOKE_DONE, "run + verify smoke test first"
    FULL_CMD = [sys.executable, DRIVER,
        "--train-file", train_path, "--val-file", val_path,
        "--output-dir", CKPT_FULL, "--tokenizer", TOKENIZER_PATH,
        "--device", "cuda", "--precision", "fp16",
        "--batch-size", "32", "--gradient-accumulation-steps", "1",
        "--learning-rate", "3e-4", "--warmup-steps", "500",
        "--epochs", "1",
        "--checkpoint-interval", "500", "--eval-interval", "500",
        "--log-interval", "50"]
    print("FULL cmd:", " ".join(FULL_CMD))
    print("est. optimizer steps ~%d (98.5M tok / (32*512)) — estimate only"
          % (98500000//(32*512)))
    import subprocess
    r = subprocess.run(FULL_CMD, cwd="/content/work")
    FULL_DONE = (r.returncode == 0)
    print("FULL_DONE:", FULL_DONE)
    print("Success = checkpoint_final.pt exists with status PRETRAINING_COMPLETED")

## Cell 12 — Checkpoint Verification

After training (or anytime): loads the latest checkpoint fresh, verifies
param count, status field, step/epoch, and that weights differ from random init.

In [ ]:
ckpts = sorted(glob.glob(os.path.join(CKPT_FULL, "checkpoint_*.pt")) +
                glob.glob(os.path.join(CKPT_SMOKE, "checkpoint_*.pt")))
print("all checkpoints:", [os.path.basename(p) for p in ckpts])
if not ckpts:
    print("no checkpoints yet — training has not produced one")
else:
    latest = ckpts[-1]
    print("latest:", latest, "%.1f MB" % (os.path.getsize(latest)/1e6))
    ck = torch.load(latest, map_location="cpu", weights_only=False)
    print("status:", ck.get("model_status"), "| step:", ck.get("step"),
          "| epoch:", ck.get("epoch"))
    print("has scaler state:", ck.get("grad_scaler_state_dict") is not None)
    print("has rng state:", ck.get("rng_state") is not None)
    print("tokenizer_path:", ck.get("tokenizer_path"))

## Cell 13 — Generation Test (honest quality check)

Loads the final checkpoint and generates short continuations for English,
Bengali, and romanized-Bengali prompts. A 100M-token English-first model will
NOT be a capable Bengali assistant — judge outputs honestly, log them.

In [ ]:
assert PACKAGE_OK
GEN_DONE = False
final_ckpt = os.path.join(CKPT_FULL, "checkpoint_final.pt")
if not os.path.exists(final_ckpt):
    cands = sorted(glob.glob(os.path.join(CKPT_FULL,"checkpoint_*.pt")))
    final_ckpt = cands[-1] if cands else None
if not final_ckpt or not os.path.exists(final_ckpt):
    print("no trained checkpoint - generation test skipped (model UNTRAINED)")
else:
    try:
        import torch as _t
        from core01.transformer import NexusAICore01
        from core01.inference import InferenceEngine
        _ck = _t.load(final_ckpt, map_location="cpu", weights_only=False)
        _model = NexusAICore01(cfg)
        _model.load_state_dict(_ck["model_state_dict"], strict=True)
        infer = InferenceEngine(_model, tok, cfg, device="cuda" if torch.cuda.is_available() else "cpu")
        prompts = ["The capital of France is", "bangladesher shohor", "ami bhate pari"]
        for prompt in prompts:
            try:
                out = infer.generate(prompt, max_new_tokens=40, do_sample=False)
                print("PROMPT:", prompt)
                print("OUT:", out)
                print("-"*40)
            except Exception as e:
                print("prompt failed:", repr(e))
        GEN_DONE = True
    except Exception as e:
        print("generation failed:", repr(e))

## Cell 14 — Drive Backup & Session Report

Copies the latest checkpoints + logs manifest to Drive (they already live
there), prints a session summary table to paste into the project
Training Session Log.

In [ ]:
import time, json
print("="*60)
print("NEXUSAI CORE-01 COLAB SESSION REPORT")
print("="*60)
for name, val in [("PACKAGE_OK", globals().get("PACKAGE_OK")),
                  ("TOKENIZER_OK", globals().get("TOKENIZER_OK")),
                  ("DATASET_OK", globals().get("DATASET_OK")),
                  ("MODEL_OK", globals().get("MODEL_OK")),
                  ("SMOKE_DONE", globals().get("SMOKE_DONE")),
                  ("FULL_DONE", globals().get("FULL_DONE")),
                  ("GEN_DONE", globals().get("GEN_DONE"))]:
    print(f"{name:15}: {val}")
for d,label in [(CKPT_SMOKE,"smoke ckpts"),(CKPT_FULL,"full ckpts")]:
    fs = glob.glob(os.path.join(d,"*.pt"))
    print(f"{label:15}: {len(fs)} files, {sum(os.path.getsize(p) for p in fs)/1e6:.0f} MB total")
print("DRIVE ckpt dir:", CKPT_FULL)
print("="*60)
print("Paste this table into README 'Training Session Log'.")
print("CORE-01 STATUS:", "TRAINED? verify checkpoint_final.pt status" if FULL_DONE else "UNTRAINED")